# /summarize — Endpoint Evaluation

For each fixture article:
1. Calls `/extract` to obtain the pre-computed extract required by `/summarize`.
2. Calls `/summarize` with `text`, `extract`, and `headline`.
3. Measures ROUGE-1 F and keyword hit rate.

**Prerequisite:** NLP service + Ollama running. `/readyz` → 200.

In [1]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, rouge1_f, keyword_hit_rate, NLP_BASE_URL, HEADERS

cases = load_fixture('summarize_cases.json')
print(f'Loaded {len(cases)} test cases')

Loaded 5 test cases


In [2]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [6]:
results = []

for case in cases:
    # Step 1: /extract
    ext_resp = requests.post(
        f'{NLP_BASE_URL}/extract',
        json={'article_id': case['article_id'], 'text': case['text']},
        headers=HEADERS,
    )
    assert ext_resp.status_code == 200, \
        f"{case['article_id']} /extract: HTTP {ext_resp.status_code} — {ext_resp.text}"
    extract = ext_resp.json()['extract']

    # Step 2: /summarize
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/summarize',
        json={
            'article_id': case['article_id'],
            'text':       case['text'],
            'extract':    extract,
            'headline':   case['raw_headline'],
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    r1       = rouge1_f(case['text'], data['summary'])
    hl_hit   = keyword_hit_rate(data['headline'], case.get('expected_headline_keywords', []))
    sum_hit  = keyword_hit_rate(data['summary'],  case.get('expected_summary_keywords', []))
    threshold = case.get('min_rouge1', 0.2)
    passed   = r1 >= threshold
    icon     = '✅' if passed else '❌'

    results.append({
        'id':                  case['article_id'],
        'rouge1':              r1,
        'headline_hit':        hl_hit,
        'summary_keyword_hit': sum_hit,
        'latency_s':           latency,
        'headline':            data['headline'],
        'summary':             data['summary'],
        'pass':                passed,
    })

    print(f"{icon} [{case['article_id']}]  ROUGE-1={r1:.3f}  "
          f"hl_hit={hl_hit:.2f}  sum_hit={sum_hit:.2f}  {latency:.1f}s")
    print(f"   headline : {data['headline']}")
    print(f"   summary  : {data['summary'][:120]}...")
    print()

✅ [sum-001]  ROUGE-1=0.599  hl_hit=1.00  sum_hit=1.00  12.8s
   headline : Madrid amplía el carril bici en la Gran Vía
   summary  : El Ayuntamiento de Madrid aprobó la ampliación del carril bici en la Gran Vía, que se extenderá por 1,3 kilómetros. Las ...

✅ [sum-002]  ROUGE-1=0.791  hl_hit=1.00  sum_hit=1.00  3.3s
   headline : Estación de Atocha estrena nuevo aparcamiento cubierto para bicicletas
   summary  : La estación de Atocha ha inaugurado un aparcamiento cubierto con capacidad para 200 plazas. La instalación cuenta además...

✅ [sum-003]  ROUGE-1=0.283  hl_hit=1.00  sum_hit=0.00  5.2s
   headline : Barcelona aprueba un ambicioso plan de movilidad sostenible para el periodo 2026-2030
   summary  : El Ayuntamiento de Barcelona ha aprobado un plan para reducir el tráfico rodado en un 40% para 2030. El proyecto incluye...

✅ [sum-004]  ROUGE-1=0.533  hl_hit=0.33  sum_hit=0.67  4.7s
   headline : Ciclista herido tras ser golpeado por vehículo en el Paseo del Prado
   summary  : Un

In [7]:
passing      = [r for r in results if r['pass']]
avg_rouge1   = sum(r['rouge1']       for r in results) / len(results)
avg_hl_hit   = sum(r['headline_hit'] for r in results) / len(results)
avg_lat      = sum(r['latency_s']    for r in results) / len(results)

print_scorecard('/summarize', {
    'Cases':                             len(results),
    'Passing (ROUGE-1 ≥ threshold)':     f'{len(passing)}/{len(results)}',
    'Avg ROUGE-1 F':                     avg_rouge1,
    'Target ROUGE-1 F':                  0.20,
    'Avg headline keyword hit':          avg_hl_hit,
    'Avg latency (s)':                   avg_lat,
})


  /summarize
  Cases                               5
  Passing (ROUGE-1 ≥ threshold)       5/5
  Avg ROUGE-1 F                       0.542
  Target ROUGE-1 F                    0.200
  Avg headline keyword hit            0.867
  Avg latency (s)                     6.312



## Tuning Guide

| Symptom | Lever | Where |
|---------|-------|-------|
| Summary too long / unfocused | Lower `max_sentences` or reduce extractive pre-step | `nlp/summarizer/service.py` |
| Extractive pre-step fires on short articles | Raise `_EXTRACTIVE_WORD_THRESHOLD` (default 500) | `nlp/summarizer/service.py` |
| LLM hallucinations | Switch `OLLAMA_MODEL` env var to a larger model | `docker-compose.yml` |
| Timeout errors | Raise `OLLAMA_TIMEOUT` env var (default 30 s) | `docker-compose.yml` |
| Headline too generic | Tighten the prompt specificity constraints | `nlp/summarizer/ollama_client.py` |